# CITE-seq and Multiome Data Integration

**Tier 3 — Applied Bioinformatics | Module 31 · Notebook 2**

*Prerequisites: Notebook 1 (scATAC-seq)*

---

**By the end of this notebook you will be able to:**
1. Describe CITE-seq (RNA + protein), 10x Multiome (RNA + ATAC), and Spatial Transcriptomics co-modality designs
2. Process ADT (antibody-derived tags) counts and perform DSB normalization
3. Perform weighted nearest neighbor (WNN) joint embedding of RNA and protein
4. Integrate matched RNA and ATAC profiles from 10x Multiome data
5. Identify cell-type-specific regulatory elements linking chromatin to expression



**Key resources:**
- [Seurat WNN tutorial](https://satijalab.org/seurat/articles/weighted_nearest_neighbor_analysis)
- [CITE-seq paper (Stoeckius et al. 2017)](https://www.nature.com/articles/nmeth.4380)
- [muon Python package](https://muon.readthedocs.io/)

---

[< Previous: Single-Cell ATAC-seq: Chromatin Accessibility](01_scatac_chromatin.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Single-Cell Batch Correction and Dataset Integr... >](03_sc_integration.ipynb)

---

## 1. CITE-seq Experimental Design

### What is CITE-seq?
CITE-seq (Cellular Indexing of Transcriptomes and Epitopes by sequencing, Stoeckius et al. 2017) simultaneously measures **RNA** and **surface protein** expression in the same cell. This is achieved by:

1. Incubating cells with a cocktail of antibodies each conjugated to a unique **DNA oligonucleotide barcode** (Feature Barcode Technology)
2. Performing standard 10x Genomics droplet capture — the antibody-DNA conjugates co-encapsulate with mRNA from the same cell
3. During library preparation, RNA and ADT (Antibody-Derived Tag) libraries are separated by size and sequenced separately

### Antibody-Derived Tags (ADT) vs Hashtag Oligos (HTO)
- **ADT**: antibodies targeting cell-surface proteins (CD3, CD4, CD8, CD19...). Measures protein expression.
- **HTO**: antibodies conjugated to sample-specific barcodes. Used to multiplex multiple samples into a single 10x run, then "demultiplex" computationally. Reduces cost; 4–16 samples per lane.

### Why measure protein AND RNA?
- **Protein is more stable**: surface protein levels reflect the "display" state of the cell, while RNA reflects production. They are correlated but capture different biology.
- **Classic markers are protein-based**: FACS-defined T cell subsets (CD4/CD8) are far cleaner by protein than RNA due to low CD4/CD8 mRNA expression.
- **Sub-population resolution**: B cell subsets (naive, memory, plasmablast) show overlapping RNA signatures but distinct protein profiles. 
- **Cross-validate clusters**: an RNA-based cluster that lacks the expected protein marker is suspect.

### Panel design considerations
- Start with known lineage markers (CD3, CD19, CD14, CD56 for PBMC)
- Include "anchoring proteins" that are highly enriched in specific cell types to validate clustering
- Include both specific and broad markers — CD45 (pan-immune) serves as a positive control
- Maximum ~200 antibodies per 10x run; prioritize functionally informative proteins
- Avoid antibodies with high non-specific background (test empirically in bulk first)

### The ADT count matrix structure
ADTs produce a separate count matrix: rows = protein targets, columns = cell barcodes. This is loaded alongside the RNA matrix in the Cell Ranger `filtered_feature_bc_matrix.h5` file, accessible as feature_type == "Antibody Capture".

## 2. ADT Processing and Normalization

### The protein background problem
ADT counts have a fundamentally different distribution from RNA:
- **High background**: even cells that don't express a protein will show hundreds of counts due to non-specific antibody binding. This creates a bimodal distribution: negative population (background ~100–500) + positive population (signal ~1000–10,000).
- **No true zeros**: unlike RNA where unexpressed genes genuinely have 0 reads, every protein shows at least background counts in every cell.
- **Log-normal, not Poisson**: the negative population follows a log-normal distribution, not the Poisson/NB model that fits RNA.

### CLR (Centered Log-Ratio) normalization
The simplest and most widely used approach. For each cell, compute the geometric mean of all ADT counts, then divide each protein count by the geometric mean and take the log:

```
CLR(x_i) = log(x_i / geometric_mean(x))
```

This centers the distribution around 0, making comparison across cells and proteins meaningful. CLR is implemented in Seurat as `NormalizeData(assay='ADT', normalization.method='CLR', margin=2)` (margin=2 = normalize across features within each cell).

**Caveat**: CLR assumes the panel is roughly balanced — if 80% of your antibodies target T cell markers and you run on B cells, the geometric mean is dominated by negatives and CLR will over-correct positives.

### DSB (Denoised and Scaled by Background) normalization
DSB (Mulè et al. 2022) is more principled:
1. Estimate per-protein background level from **empty droplets** (droplets with no cell but still containing ambient antibody)
2. Subtract the expected background from each protein's count
3. Divide by the technical noise component (estimated from the data)

**Result**: DSB-normalized values are interpretable as signal-to-noise ratios above background. Values > 3–4 indicate a clearly positive cell.

**Requirement**: you need empty droplet data (from Cell Ranger `raw_feature_bc_matrix.h5`, not just the filtered version).

### When to use which normalization
| Method | Needs empty droplets | Speed | Best for |
|--------|---------------------|-------|---------|
| CLR | No | Fast | Quick analysis, <50 proteins |
| DSB | Yes | Moderate | Rigorous analysis, high background proteins |
| Raw log | No | Fastest | Initial exploration only |

In [ ]:
# ADT normalization: CLR and DSB-style approaches with visualization
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# --- CLR normalization ---
def clr_normalize(X):
    """Centered log-ratio normalization for ADT counts.
    X: cells x proteins matrix (raw counts, no zeros assumed but we add pseudocount)
    """
    # Add pseudocount to avoid log(0)
    X_ps = X + 0.5
    # Geometric mean per cell (axis=1 = across proteins within a cell)
    geo_mean = np.exp(np.log(X_ps).mean(axis=1, keepdims=True))
    return np.log(X_ps / geo_mean)

adt_clr = clr_normalize(adt_counts)

# --- DSB-style normalization (simplified without empty droplets) ---
# Simulate empty droplet background: lower count range
n_empty = 200
empty_adt = rng.poisson(80, (n_empty, n_proteins)).astype(np.float32)  # ambient only

def dsb_normalize(X, background):
    """Simplified DSB normalization.
    Subtracts per-protein background mean, scales by background std.
    """
    bg_mean = np.log1p(background).mean(axis=0)
    bg_std  = np.log1p(background).std(axis=0) + 1e-6
    return (np.log1p(X) - bg_mean) / bg_std

adt_dsb = dsb_normalize(adt_counts, empty_adt)

print("CLR normalized ADT:")
print(f"  Range: [{adt_clr.min():.2f}, {adt_clr.max():.2f}]")
print(f"  Mean: {adt_clr.mean():.3f}")

print("\nDSB normalized ADT:")
print(f"  Range: [{adt_dsb.min():.2f}, {adt_dsb.max():.2f}]")
print(f"  DSB > 3 (positive cells): {(adt_dsb > 3).mean():.3f}")

# Visualize: bimodal distribution for CD3 protein
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Raw counts for CD3 and CD4
for col, prot_idx, prot_name in [(0, 0, 'CD3'), (1, 1, 'CD4'), (2, 3, 'CD19')]:
    raw_vals = np.log1p(adt_counts[:, prot_idx])
    clr_vals = adt_clr[:, prot_idx]
    dsb_vals = adt_dsb[:, prot_idx]
    
    axes[0, col].hist(raw_vals, bins=40, edgecolor='k', alpha=0.7, color='steelblue')
    axes[0, col].set_title(f'{prot_name}: log1p raw counts')
    axes[0, col].set_xlabel('log1p(ADT count)')
    axes[0, col].set_ylabel('Cells')
    
    axes[1, col].hist(dsb_vals, bins=40, edgecolor='k', alpha=0.7, color='tomato')
    axes[1, col].axvline(3, color='black', linestyle='--', label='DSB=3 (positive cutoff)')
    axes[1, col].set_title(f'{prot_name}: DSB normalized')
    axes[1, col].set_xlabel('DSB score')
    axes[1, col].legend(fontsize=8)

plt.suptitle('ADT distribution: raw counts vs DSB normalization\n'
             'Bimodal distribution indicates positive/negative cell populations', y=1.02)
plt.tight_layout()
plt.savefig('adt_normalization.png', dpi=100, bbox_inches='tight')
plt.show()

# Show that CLR-normalized values separate cell types for CD3
print("\nCD3 CLR values by cell type:")
for ct in ['CD4_T', 'CD8_T', 'B_cell', 'Monocyte', 'NK_cell']:
    mask = np.array(all_labels) == ct
    print(f"  {ct}: mean={adt_clr[mask, 0].mean():.2f}  (positive should be T/NK cells)")

## 3. Weighted Nearest Neighbor (WNN) Integration

### The core problem: combining two noisy views
RNA and protein measure partially overlapping cell identity signals. Naively concatenating normalized RNA and ADT matrices and running PCA would be dominated by RNA (2000+ features vs 20 proteins). A simple average of RNA and protein distances would give equal weight regardless of which modality is more informative for a given cell.

**WNN (Hao et al. 2021, Cell)** solves this by learning per-cell, per-modality weights:
- In cells where RNA gives a confident, well-clustered position → RNA gets higher weight
- In cells where RNA is ambiguous but protein is clear → protein gets higher weight

The result is a **within-cell adaptive weighting** that extracts the most information from each cell across modalities.

### WNN algorithm steps
1. Compute a **within-modality k-NN graph** (RNA k-NN and protein k-NN independently)
2. For each cell, estimate the **predictive power** of each modality by asking: how well can RNA neighbors predict protein expression, and vice versa?
3. Combine k-NN graphs using per-cell weights: 
   `WNN_affinity(i,j) = w_RNA(i) * RNA_affinity(i,j) + w_protein(i) * protein_affinity(i,j)`
4. UMAP is run on the WNN graph (not on individual modality embeddings)

### Why WNN is superior for CITE-seq
- **T cell sub-typing**: CD4 mRNA is low-expressed and noisy; CD4 protein is clear. WNN gives protein high weight for T cells.
- **Plasma cells**: very high RNA output (immunoglobulins), distinct from memory B cells in RNA space, but both are CD19+. RNA dominates appropriately here.
- **Monocyte/DC boundary**: CD11c protein distinguishes plasmacytoid DC from myeloid DC; RNA doesn't always separate them cleanly.

### muon Python implementation
```python
import muon as mu

# MuData object holds multiple AnnData modalities
mdata = mu.MuData({'rna': adata_rna, 'prot': adata_prot})

# RNA: standard scanpy preprocessing
mu.pp.filter_var(mdata['rna'], 'n_cells_by_counts', lb=10)
sc.pp.normalize_total(mdata['rna'], target_sum=1e4)
sc.pp.log1p(mdata['rna'])
sc.pp.pca(mdata['rna'], n_comps=30)
sc.pp.neighbors(mdata['rna'])

# Protein: CLR normalization + PCA
mu.prot.pp.clr(mdata['prot'])
sc.pp.pca(mdata['prot'], n_comps=15)
sc.pp.neighbors(mdata['prot'])

# WNN integration
mu.pp.neighbors(mdata, key_added='wnn')
sc.tl.umap(mdata, neighbors_key='wnn')
sc.pl.umap(mdata, color='leiden_wnn')
```

## 4. 10x Multiome: Paired RNA + ATAC from the Same Cell

### What is 10x Multiome?
The 10x Chromium Multiome ATAC + Gene Expression kit captures both **nuclear RNA** and **chromatin accessibility** from the same single nucleus in one experiment. This is fundamentally different from CITE-seq (RNA + protein) — here you get two views of the regulatory genome: what's accessible AND what's being transcribed.

### Key differences from CITE-seq
| Aspect | CITE-seq | 10x Multiome |
|--------|----------|-------------|
| Modalities | RNA + protein | RNA + ATAC |
| Cell vs nucleus | Cells (cytoplasm+nucleus) | Nuclei only |
| Protein panels | Up to ~200 targets | N/A |
| Regulatory inference | Limited | Direct (peak → gene) |
| Read requirement | 20k/cell RNA + 1k/cell ADT | 20k/cell RNA + 10k/cell ATAC |

### Why paired RNA+ATAC is powerful
Standard scATAC-seq gives you open chromatin peaks, but assigning these peaks to regulatory target genes requires assumptions. With paired RNA+ATAC from the same cells:
- Directly correlate peak accessibility with target gene expression (no assumptions)
- Identify **cell-type-specific regulatory elements**: peaks open in cell type A that correlate with a gene highly expressed in cell type A
- **TF footprinting validation**: find TF motif in open peak AND see TF expression in same cell
- **Gene regulation directionality**: peak opens before gene activates → regulatory (not reactive)

### muon for Multiome analysis
```python
import muon as mu

# Load 10x Multiome h5 file (contains both RNA and ATAC)
mdata = mu.read_10x_h5('filtered_feature_bc_matrix.h5')
# mdata['rna']  — AnnData with RNA counts
# mdata['atac'] — AnnData with peak counts

# Standard RNA preprocessing
mu.pp.filter_var(mdata['rna'], 'n_cells_by_counts', lb=10)
sc.pp.normalize_total(mdata['rna'])
sc.pp.log1p(mdata['rna'])
sc.pp.pca(mdata['rna'])

# ATAC: TF-IDF + LSI (drop first component)
mu.atac.pp.tfidf(mdata['atac'], scale_factor=1e4)
sc.tl.pca(mdata['atac'])  # LSI component 1 will be depth-correlated

# Joint UMAP (WNN concept applied to RNA + ATAC)
mu.pp.neighbors(mdata)
sc.tl.umap(mdata)
```

### Gene activity scores (bridging ATAC to RNA)
In the absence of paired RNA, scATAC clusters can be annotated using **gene activity scores**: sum the ATAC signal in the gene body + promoter (2kb upstream) for each gene, creating a pseudo-expression matrix. This proxy expression is much noisier than actual RNA but enables:
- Coarse cell type annotation using well-known marker genes
- Reference mapping to RNA atlases for label transfer

## 5. Cross-Modality Visualization

### Visualizing protein expression per cluster
After WNN-based clustering, it is informative to show protein expression levels for each cell type. This serves both as a validation (do known markers show expected patterns?) and as a discovery tool.

**Violin plots** show the full distribution of protein expression per cell type — important because bimodal distributions (positive/negative populations) may be hidden in a mean heatmap.

**Heatmaps of mean expression** give a compact overview — each row is a protein, each column is a cluster.

### When RNA and protein disagree
Discordances between RNA and protein are biologically meaningful:
- **High protein, low RNA**: protein has long half-life (CD27, CD45RO on memory T cells) — reflects past activation, not current transcription
- **High RNA, low protein**: active transcription but post-translational degradation or secretion (cytokines like IFN-gamma)
- **Unexpected protein+**: contamination / doublets / non-specific antibody binding — investigate with QC plots

### What to report in publications
For CITE-seq papers:
1. UMAP colored by WNN clustering (not RNA-only)
2. Per-protein violin plots for key lineage markers
3. Heatmap of all ADT proteins x clusters
4. Modality weight plots showing where RNA vs protein dominates
5. Concordance analysis between RNA marker genes and corresponding proteins

In [ ]:
# Cross-modality visualization: protein heatmap, violin plots, concordance
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

cell_types_list = ['CD4_T', 'CD8_T', 'B_cell', 'Monocyte', 'NK_cell']
labels_arr = np.array(all_labels)

# --- Heatmap: mean CLR protein per cell type ---
mean_prot = np.zeros((len(cell_types_list), n_proteins))
for i, ct in enumerate(cell_types_list):
    mask = labels_arr == ct
    mean_prot[i, :] = adt_clr[mask].mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

im = axes[0].imshow(mean_prot, aspect='auto', cmap='RdBu_r',
                    vmin=-3, vmax=3)
axes[0].set_xticks(range(n_proteins))
axes[0].set_xticklabels(protein_names, rotation=45, ha='right', fontsize=8)
axes[0].set_yticks(range(len(cell_types_list)))
axes[0].set_yticklabels(cell_types_list)
axes[0].set_title('Mean CLR-normalized protein expression per cell type')
plt.colorbar(im, ax=axes[0], label='CLR expression', shrink=0.8)

# --- Violin plots for key markers (CD3, CD4, CD19, CD14) ---
key_proteins = ['CD3', 'CD4', 'CD19', 'CD14']
key_indices  = [protein_names.index(p) for p in key_proteins]
positions = range(len(cell_types_list))

violin_data_matrix = []
for prot_name, prot_idx in zip(key_proteins, key_indices):
    data_by_ct = [adt_clr[labels_arr == ct, prot_idx] for ct in cell_types_list]
    violin_data_matrix.append(data_by_ct)

# Plot violins for each protein
fig2, axes2 = plt.subplots(1, 4, figsize=(20, 5))
colors_map = {'CD4_T':'#e41a1c','CD8_T':'#ff7f00','B_cell':'#377eb8',
              'Monocyte':'#4daf4a','NK_cell':'#984ea3'}
ct_colors = [colors_map[ct] for ct in cell_types_list]

for ax, prot_name, data_by_ct in zip(axes2, key_proteins, violin_data_matrix):
    parts = ax.violinplot(data_by_ct, positions=range(len(cell_types_list)),
                         showmedians=True, showextrema=False)
    for i, (pc, color) in enumerate(zip(parts['bodies'], ct_colors)):
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
    ax.set_xticks(range(len(cell_types_list)))
    ax.set_xticklabels([ct[:5] for ct in cell_types_list], rotation=30, ha='right')
    ax.set_title(f'{prot_name} protein\n(CLR normalized)')
    ax.set_ylabel('CLR value')
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Protein expression distributions by cell type\n'
             'High CLR values = protein positive; negative/near-zero = background', y=1.02)
plt.tight_layout()
plt.savefig('cite_seq_violins.png', dpi=100, bbox_inches='tight')
plt.show()

# --- RNA-protein concordance for matching markers ---
# Compare CD3 mRNA (sum of CD3D/CD3E-like genes at offsets 0-2) vs CD3 protein
cd3_rna_proxy = rna_norm[:, :10].sum(axis=1)   # early T cell genes as proxy
cd3_prot = adt_clr[:, 0]  # CD3 protein

from scipy.stats import pearsonr, spearmanr
r_pearson, _ = pearsonr(cd3_rna_proxy, cd3_prot)
r_spearman, _ = spearmanr(cd3_rna_proxy, cd3_prot)

print(f"CD3 RNA vs protein concordance:")
print(f"  Pearson r  = {r_pearson:.3f}")
print(f"  Spearman r = {r_spearman:.3f}")
print(f"\nNote: RNA-protein correlation is typically 0.3-0.7 for surface markers")
print(f"Lower correlation = more information gained by measuring both modalities")

# Scatter plot
fig3, ax = plt.subplots(figsize=(6, 5))
for ct in cell_types_list:
    mask = labels_arr == ct
    ax.scatter(cd3_rna_proxy[mask], cd3_prot[mask], c=colors_map[ct],
               s=10, alpha=0.6, label=ct)
ax.set_xlabel('CD3-like RNA (log-normalized)')
ax.set_ylabel('CD3 protein (CLR)')
ax.set_title(f'RNA vs protein concordance (r={r_pearson:.2f})\nDiscordant cells may reflect post-translational regulation')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('rna_protein_concordance.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. Weighted Nearest Neighbor (WNN) Integration

> WNN concept: per-cell adaptive weighting of RNA vs protein modalities. Build WNN graph and compute WNN-UMAP. Cell types resolved by protein markers.

## 4. 10x Multiome RNA + ATAC Integration

> Paired per-cell RNA + ATAC from same cells. Low-dimensional joint embedding. Peak-gene correlation in the same cell. Regulatory inference.

## 5. Cross-Modality Visualization

> Side-by-side UMAP: RNA-based vs protein-based vs WNN. Per-protein violin plots per cell type. Linking gene expression to chromatin accessibility for same cells.

---

[< Previous: Single-Cell ATAC-seq: Chromatin Accessibility](01_scatac_chromatin.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Single-Cell Batch Correction and Dataset Integr... >](03_sc_integration.ipynb)

---

## Summary

**Key concepts from this notebook:**

| Concept | Core idea | Implementation |
|---------|-----------|---------------|
| CITE-seq | RNA + protein from same cell | 10x Feature Barcoding |
| ADT normalization | Remove background signal | CLR (quick) or DSB (rigorous) |
| Bimodal distribution | Background vs signal populations | DSB cutoff ~3 |
| WNN | Per-cell adaptive modality weighting | Seurat v4+ / muon |
| 10x Multiome | RNA + ATAC from same nucleus | muon/episcanpy |
| Gene activity | ATAC → pseudo-expression | Sum peaks in gene body |

**Common pitfalls:**
- **Applying log1p before CLR**: CLR should be applied to raw counts, not log-normalized data
- **Using RNA-only UMAP for CITE-seq**: WNN gives better cluster separation for immune cells
- **Ignoring ADT QC**: proteins with > 80% cells positive (anti-isotype contamination) or < 5% cells positive (failed antibody) should be removed
- **Confusing Multiome with CITE-seq**: Multiome gives DNA accessibility, not protein expression — completely different biology

**Downstream applications:**
- WNN clusters → cell type annotation using known protein markers
- Peak-gene links (Multiome) → regulatory network inference
- TF motif activity (chromVAR on ATAC) + TF expression (RNA) → TF binding validation
- Differential accessibility between cell types → ATAC-based marker peaks